# 🏗️ Building a Mini Virtual Workforce
## From Tools (MCP) to Multi-Agent Delegation (A2A)

**Workshop Duration:** 40 min hands-on &nbsp;|&nbsp; **Level:** Intermediate Python &nbsp;|&nbsp; **Runtime:** Google Colab (CPU)

---

### 🎯 What You'll Build

```
 ┌──────────────────────────────────────────────────────────────┐
 │                      USER REQUEST                            │
 │          "Plan a cloud migration strategy"                   │
 └──────────────────────────┬───────────────────────────────────┘
                            │
                            ▼
 ┌──────────────────────────────────────────────────────────────┐
 │                  🎯 ORCHESTRATOR AGENT                       │
 │            (Coordinates the virtual workforce)               │
 │                                                              │
 │   ┌────────────────────────┐   ┌──────────────────────────┐  │
 │   │ 🔧 MCP TOOL LAYER     │   │ 🤖 A2A AGENT LAYER      │  │
 │   │ (Bounded, Deterministic│   │ (Unbounded, Reasoning)   │  │
 │   │                        │   │                          │  │
 │   │ query_cloud_benchmarks │   │ EnterpriseArchSpecialist │  │
 │   │ → Structured JSON data │   │ → Reasoned analysis      │  │
 │   └────────────────────────┘   └──────────────────────────┘  │
 │                                                              │
 │                    📋 FINAL STRATEGY                         │
 └──────────────────────────────────────────────────────────────┘
```

### 📚 Learning Objectives

| # | Concept | What You'll Do |
|---|---------|----------------|
| 1 | **Agent = Model + Harness** | Build the mental model & core data structures |
| 2 | **MCP Tools** | Create a bounded, deterministic data retrieval tool |
| 3 | **A2A Delegation** | Build a specialist agent with an Agent Card & task schema |
| 4 | **Orchestration Pipeline** | Wire MCP + A2A into a cohesive multi-agent workflow |
| 5 | **Hands-on Challenge** | Wire a `SecurityAuditSpecialist` with automated verification (10 min) |

---
## ⚙️ Environment Setup

Run the cell below to install dependencies. Everything runs on **CPU** — no GPU or paid API key needed.

In [ ]:
# Install dependencies (only google-genai is needed if using Gemini toggle)
!pip install -q google-genai 2>/dev/null || echo "google-genai not installed (OK for mock mode)"
print("✅ Setup complete!")

In [ ]:
from __future__ import annotations

# ============================================================
# 🔧 CONFIGURATION — Toggle LLM Backend
# ============================================================
# Set USE_GEMINI = True and provide an API key to use Gemini.
# Set USE_GEMINI = False to run entirely with the built-in Mock LLM.
# The Mock LLM produces realistic canned responses — no API key needed!

USE_GEMINI = False                        # ← Toggle here
GEMINI_API_KEY = ""                       # ← Paste key here (or set GOOGLE_API_KEY env var)
GEMINI_MODEL = "gemini-2.0-flash"        # ← Model name if using Gemini

# ============================================================
# 📦 Imports
# ============================================================

import json
import os
import textwrap
import time
import uuid
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum
from typing import Any, Callable, Dict, List, Optional

print(f"🔧 LLM Backend: {'Google Gemini (' + GEMINI_MODEL + ')' if USE_GEMINI else 'Mock LLM (no API key needed)'}")
print("✅ All imports loaded.")

---
# 📖 Section 1: THE FOUNDATION — Agent = Model + Harness

> **Core insight:** An agent is NOT just an LLM. An agent is a **Model** (the brain)
> wrapped in a **Harness** (the runtime body) that gives it tools, memory, protocols, and control loops.

```
 ┌─────────────────────────────────────────────────┐
 │                   A G E N T                     │
 │                                                 │
 │  ┌───────────────────────────────────────────┐  │
 │  │            🧠  MODEL  (LLM)              │  │
 │  │   Reasoning · Planning · Language         │  │
 │  └───────────────────────────────────────────┘  │
 │                                                 │
 │  ┌───────────────────────────────────────────┐  │
 │  │          🦾  HARNESS  (Runtime)           │  │
 │  │                                           │  │
 │  │   🔧 Tools    📋 Memory    🔒 Guardrails │  │
 │  │   📡 Protocols (MCP, A2A)                │  │
 │  └───────────────────────────────────────────┘  │
 └─────────────────────────────────────────────────┘
```

### The Integration Problem: O(N × M) → O(N + M)

Without standard protocols, every agent framework needs custom glue code for every tool:

```
 WITHOUT PROTOCOLS:  O(N × M)          WITH PROTOCOLS:  O(N + M)
 ─────────────────────────────          ──────────────────────────────
                                        
 Agent A ──┬── Tool 1                   Agent A ──┐
           ├── Tool 2                   Agent B ──┼── Standard Client
           └── Tool 3                   Agent C ──┘
 Agent B ──┬── Tool 1                              │
           ├── Tool 2                        ┌─────┴─────┐
           └── Tool 3                        │ PROTOCOL  │
 Agent C ──┬── Tool 1                        │ (MCP/A2A) │
           ├── Tool 2                        └─────┬─────┘
           └── Tool 3                              │
                                        Tool 1 ───┤
 Integrations: 3 × 3 = 9               Tool 2 ───┼── Standard Server
                                        Tool 3 ───┘
                                        
                                        Integrations: 3 + 3 = 6
```

- **MCP (Model Context Protocol)**: Standardizes **Agent ↔ Tool / Data** communication.
- **A2A (Agent-to-Agent)**: Standardizes **Agent ↔ Agent** communication and task delegation.

In [ ]:
# ============================================================
# 📦 SECTION 1: Core Data Models
# ============================================================
# These cleanly typed dataclasses define the vocabulary of our mini framework.
# They mirror real MCP/A2A protocol specifications in simplified form.

# --- Enumerations ---

class TaskState(Enum):
    """Lifecycle states for an A2A task."""
    SUBMITTED  = "submitted"
    WORKING    = "working"
    COMPLETED  = "completed"
    FAILED     = "failed"


class MessageRole(Enum):
    """Sender role in an A2A conversation."""
    USER  = "user"
    AGENT = "agent"


# --- MCP Tool Models ---

@dataclass
class ToolParameter:
    """Parameter definition for an MCP tool — mirrors JSON Schema."""
    name: str
    type: str
    description: str
    required: bool = True


@dataclass
class MCPToolSpec:
    """
    Full specification of an MCP-compliant tool.
    Discovered by agents connecting to an MCP server.
    """
    name: str
    description: str
    parameters: List[ToolParameter]
    handler: Callable[..., Dict[str, Any]]


# --- A2A Protocol Models ---

@dataclass
class AgentCard:
    """
    An A2A Agent Card — the machine-readable 'resume' of a specialist.
    Published at /.well-known/agent.json so orchestrators can discover
    capabilities, skills, and endpoints.
    """
    name: str
    description: str
    url: str
    version: str = "1.0"
    capabilities: List[str] = field(default_factory=list)
    skills: List[str] = field(default_factory=list)

    def display(self):
        print(f"┌{'─' * 56}┐")
        print(f"│ 🪪  Agent Card: {self.name:<38} │")
        print(f"├{'─' * 56}┤")
        print(f"│ Desc:   {self.description[:46]:<46} │")
        print(f"│ URL:    {self.url[:46]:<46} │")
        print(f"│ Skills: {', '.join(self.skills)[:46]:<46} │")
        print(f"└{'─' * 56}┘")


@dataclass
class A2AMessage:
    """A message exchanged during an A2A task session."""
    role: MessageRole
    content: str
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())


@dataclass
class A2ATask:
    """
    An A2A Task — the asynchronous stateful unit of work delegated across agents.
    """
    id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])
    state: TaskState = TaskState.SUBMITTED
    messages: List[A2AMessage] = field(default_factory=list)
    artifacts: List[Dict[str, Any]] = field(default_factory=list)

    def add_message(self, role: MessageRole, content: str):
        self.messages.append(A2AMessage(role=role, content=content))

    def complete(self, result: str):
        self.state = TaskState.COMPLETED
        self.artifacts.append({"type": "text", "content": result})


print("✅ Core data models defined: ToolParameter, MCPToolSpec, AgentCard, A2AMessage, A2ATask")

In [ ]:
# ============================================================
# 🧠 LLM Abstraction Layer
# ============================================================

class BaseLLM:
    """Abstract base interface for LLMs."""
    def generate(self, prompt: str) -> str:
        raise NotImplementedError


class MockLLM(BaseLLM):
    """
    A deterministic, zero-dependency Mock LLM for offline workshops.
    Produces high-fidelity, realistic responses based on prompt keywords.
    """
    def generate(self, prompt: str) -> str:
        p = prompt.lower()

        # Orchestrator synthesis
        if "synthesize" in p or ("strategy" in p and "benchmark" in p):
            return (
                "## Synthesized Cloud Migration Strategy\n\n"
                "### 1. Compute & Serving Architecture\n"
                "- Deploy stateless services to **Cloud Run / Container Engine** with automated auto-scaling.\n"
                "- Enforce API Gateway ingress with rate limiting and JWT validation.\n\n"
                "### 2. Distributed Data Layer\n"
                "- Migrate transactional core to **Cloud Spanner** (evaluated: p50=6.1ms latency, 99.999% SLA).\n"
                "- Implement managed Redis cluster for sub-millisecond caching of hot catalog data.\n\n"
                "### 3. Execution Roadmap\n"
                "- **Phase 1 (Weeks 1-4)**: Deploy Strangler Fig facade and migrate read workloads.\n"
                "- **Phase 2 (Weeks 5-8)**: Migrate stateful order processing to distributed SQL.\n"
                "- **Phase 3 (Weeks 9-12)**: Decommission legacy monolith.\n\n"
                "_[Generated via MockLLM — toggle USE_GEMINI=True for live API inference]_"
            )

        # Enterprise Architecture Specialist
        if "architect" in p or "enterprise" in p or "migration" in p:
            return (
                "## Enterprise Architecture Analysis\n\n"
                "**Assessment**: The current monolithic architecture presents scaling bottlenecks and tight coupling.\n\n"
                "**Key Recommendations**:\n"
                "1. **Pattern**: Apply the **Strangler Fig Pattern** to extract high-traffic domains incrementally.\n"
                "2. **Data Isolation**: Decompose monolithic DB into domain data stores; use Cloud Spanner for global ACID consistency.\n"
                "3. **Messaging**: Establish asynchronous event bus (Cloud Pub/Sub) for inter-service workflows.\n"
                "4. **Resiliency**: Implement circuit breakers and graceful degradation on payment and checkout boundaries.\n\n"
                "_[Specialist Analysis generated via MockLLM]_"
            )

        # Security Audit Specialist
        if "security" in p or "audit" in p or "vulnerab" in p:
            return (
                "## Security & Compliance Audit Report\n\n"
                "**Overall Risk Rating**: MEDIUM-HIGH\n\n"
                "**Identified Vulnerabilities & Risks**:\n"
                "1. ⚠️ **Lateral Movement Risk**: Monolith lacks service-to-service mutual TLS (mTLS).\n"
                "2. ⚠️ **Credential Exposure**: Hardcoded DB connection strings detected in legacy environment configs.\n"
                "3. ⚠️ **Public Ingress**: Missing Web Application Firewall (WAF) & DDoS mitigation at ingress.\n"
                "4. ✅ **Data at Rest**: Storage encryption compliant with AES-256 standard.\n\n"
                "**Remediation Requirements**:\n"
                "- Mandate Workload Identity & Secret Manager integration prior to cutover.\n"
                "- Require mTLS service mesh for all east-west microservice communication.\n\n"
                "_[Security Audit generated via MockLLM]_"
            )

        # Default fallback
        return f"Assessment based on provided context:\n\nPrompt processed successfully.\n\n_[MockLLM Generic Response]_"


class GeminiLLM(BaseLLM):
    """Google Gemini backend via google-genai SDK."""
    def __init__(self, api_key: str, model: str = "gemini-2.0-flash"):
        from google import genai
        self.client = genai.Client(api_key=api_key)
        self.model = model
        print(f"  🌐 Gemini client initialized (model={model})")

    def generate(self, prompt: str) -> str:
        response = self.client.models.generate_content(
            model=self.model,
            contents=prompt,
        )
        return response.text


# Instantiate configured backend
if USE_GEMINI:
    api_key = GEMINI_API_KEY or os.environ.get("GOOGLE_API_KEY", "")
    if not api_key:
        raise ValueError("❌ Set GEMINI_API_KEY or GOOGLE_API_KEY env var!")
    llm = GeminiLLM(api_key=api_key, model=GEMINI_MODEL)
else:
    llm = MockLLM()

print(f"✅ LLM Backend Ready: {type(llm).__name__}")

---
# 🔧 Section 2: THE BOUNDED LAYER — MCP Tools

> **Key Principle:** Tools are **strictly bounded, deterministic, and fire-and-forget**.
> They do one job, return structured data (JSON), maintain no reasoning loops, and have no open-ended conversational state.

### Bounded Tools vs Unbounded Agents

| Dimension | MCP Tool (Bounded) | A2A Agent (Unbounded) |
|---|---|---|
| **Scope** | Single deterministic operation | Multi-step reasoning & planning |
| **Output** | Structured JSON | Rich natural language & artifacts |
| **State** | Stateless / Fire-and-forget | Stateful conversation & task memory |
| **Execution** | Deterministic code execution | Non-deterministic LLM synthesis |
| **Search Space** | Fixed parameter schema | Open-ended solution space |

### The MCP Request-Response Lifecycle

```
 Agent (Client)                    MCP Server
      │                                │
      │─── 1. Discover tools ─────────►│  (tools/list)
      │◄── 2. Return tool specs ───────│
      │                                │
      │─── 3. Call tool(args) ────────►│  (tools/call)
      │◄── 4. Return result JSON ──────│
      │                                │
      ▼                                ▼
  No state stored                 No reasoning loops
```

In [ ]:
# ============================================================
# 🔧 SECTION 2: Define an MCP Tool & Tool Server
# ============================================================

def query_cloud_benchmarks(service_name: str) -> Dict[str, Any]:
    """
    Simulates an MCP tool querying a cloud performance database.
    
    Properties of this Bounded Tool:
      ✅ Deterministic: Same input yields exact same benchmark metric.
      ✅ Stateless: Does not track who called it or previous queries.
      ✅ Structured: Returns clean JSON schemas, not ambiguous text.
    """
    benchmarks = {
        "cloud-sql": {
            "service": "Cloud SQL (PostgreSQL Enterprise)",
            "latency_p50_ms": 4.2,
            "latency_p99_ms": 18.7,
            "throughput_qps": 15000,
            "availability_sla": "99.95%",
            "monthly_cost_estimate_usd": 450,
            "region": "us-central1"
        },
        "cloud-spanner": {
            "service": "Cloud Spanner (Multi-Region)",
            "latency_p50_ms": 6.1,
            "latency_p99_ms": 22.3,
            "throughput_qps": 50000,
            "availability_sla": "99.999%",
            "monthly_cost_estimate_usd": 2200,
            "region": "nam6 (multi-region)"
        },
        "bigquery": {
            "service": "BigQuery Analytics Engine",
            "latency_p50_ms": 1200,
            "latency_p99_ms": 8500,
            "throughput_qps": 200,
            "availability_sla": "99.99%",
            "monthly_cost_estimate_usd": 800,
            "region": "US multi-region"
        },
        "cloud-run": {
            "service": "Cloud Run Serverless Compute",
            "latency_p50_ms": 52,
            "latency_p99_ms": 340,
            "throughput_qps": 10000,
            "availability_sla": "99.95%",
            "monthly_cost_estimate_usd": 180,
            "region": "us-central1"
        }
    }

    key = service_name.lower().replace(" ", "-").replace("_", "-")
    if key in benchmarks:
        return {"status": "ok", "data": benchmarks[key]}
    else:
        return {
            "status": "error",
            "message": f"Unknown cloud service: '{service_name}'",
            "available_services": list(benchmarks.keys())
        }


# Register the MCP Tool Spec
cloud_benchmark_tool = MCPToolSpec(
    name="query_cloud_benchmarks",
    description="Retrieve latency, throughput, SLA, and pricing benchmarks for a cloud service.",
    parameters=[
        ToolParameter(
            name="service_name",
            type="string",
            description="The cloud service key (e.g., 'cloud-sql', 'cloud-spanner', 'cloud-run')"
        )
    ],
    handler=query_cloud_benchmarks
)


class MCPToolServer:
    """
    In-memory MCP Server implementation simulating MCP stdio/SSE protocol endpoints.
    """
    def __init__(self):
        self._tools: Dict[str, MCPToolSpec] = {}

    def register_tool(self, tool: MCPToolSpec):
        self._tools[tool.name] = tool

    def list_tools(self) -> List[Dict[str, Any]]:
        """MCP tools/list endpoint"""
        return [
            {
                "name": t.name,
                "description": t.description,
                "parameters": [
                    {"name": p.name, "type": p.type, "description": p.description, "required": p.required}
                    for p in t.parameters
                ]
            }
            for t in self._tools.values()
        ]

    def call_tool(self, name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
        """MCP tools/call endpoint"""
        if name not in self._tools:
            return {"status": "error", "message": f"Tool '{name}' not found on server"}
        try:
            return self._tools[name].handler(**arguments)
        except Exception as e:
            return {"status": "error", "message": str(e)}


# Initialize Server
mcp_server = MCPToolServer()
mcp_server.register_tool(cloud_benchmark_tool)

print(f"✅ MCP Server initialized with tool: {cloud_benchmark_tool.name}")

In [ ]:
# ============================================================
# 🧪 Single-Agent MCP Test: Discovery & Execution
# ============================================================
print("=" * 60)
print("  🧪 SINGLE-AGENT TEST: MCP Tool Call")
print("=" * 60)

# Step 1: Discover Tools
print("\n📡 Step 1: Agent queries MCP Server (tools/list)...")
tools = mcp_server.list_tools()
for t in tools:
    print(f"   🔧 Discovered: {t['name']} - {t['description']}")

# Step 2: Invoke Tool
print("\n⚡ Step 2: Agent executes MCP Tool (tools/call)...")
target_service = "cloud-spanner"
result = mcp_server.call_tool("query_cloud_benchmarks", {"service_name": target_service})
print(f"   Status: {result['status']}")
print(f"   Output JSON: {json.dumps(result.get('data', result), indent=4)}")

# Step 3: LLM Reasoning Over Tool Output
print("\n🧠 Step 3: Agent reasons over structured facts...")
prompt = (
    f"Analyze these cloud benchmark metrics for a high-traffic e-commerce database:\n"
    f"{json.dumps(result.get('data', {}), indent=2)}\n"
    f"State whether this meets 99.99% availability requirements."
)
response = llm.generate(prompt)
print(f"   Agent Output:\n   {response.strip()[:250]}...")
print("\n✅ Single-Agent MCP Test Completed Successfully!")

---
# 🤖 Section 3: THE UNBOUNDED LAYER — A2A Specialists

### Why Complex Reasoning Breaks Single Agents

When an application grows, loading every domain responsibility into a single prompt causes **The Single-Agent Ceiling**:

```
 ┌──────────────────────────────────────────────────────────────┐
 │              THE SINGLE-AGENT CEILING                        │
 │                                                              │
 │  1️⃣  CONTEXT OVERLOAD                                       │
 │     Too many competing system instructions degrade reasoning  │
 │     quality and cause hallucination.                         │
 │                                                              │
 │  2️⃣  TOOL SEARCH SPACE EXPLOSION                            │
 │     Providing 50+ tool schemas causes the LLM to misfire or   │
 │     choose incorrect tools.                                  │
 │                                                              │
 │  3️⃣  LACK OF DOMAIN ENCAPSULATION                           │
 │     Security, architecture, and finance require distinct     │
 │     evaluation standards and isolated verification loops.    │
 └──────────────────────────────────────────────────────────────┘
```

### The Solution: Agent-to-Agent (A2A) Delegation

A2A allows a central **Orchestrator** to delegate open-ended sub-tasks to autonomous **Specialist Agents**:

```
 Orchestrator Agent (Client)          EnterpriseArchSpecialist (Server)
      │                                     │
      │── 1. GET /.well-known/agent.json ──►│  (Agent Card Discovery)
      │◄── 2. Return AgentCard ─────────────│
      │                                     │
      │── 3. POST /tasks/send ─────────────►│  (Delegate Task)
      │◄── 4. Return Task (state=working) ──│
      │                                     │
      │── 5. GET /tasks/{id} ──────────────►│  (Poll / Await Completion)
      │◄── 6. Return Task (state=completed)─│
      │                                     │
      ▼                                     ▼
 Synthesizes Final Strategy            Autonomous Domain Reasoning
```

In [ ]:
# ============================================================
# 🤖 SECTION 3: Define an A2A Specialist Agent
# ============================================================

class A2ASpecialistAgent:
    """
    A domain-specialized agent conforming to A2A protocol patterns.
    
    Encapsulates:
      - Agent Card (Metadata, capabilities, skills)
      - Specialized System Prompt
      - Task Registry (Tracking asynchronous task states & artifacts)
    """
    def __init__(self, card: AgentCard, llm_backend: BaseLLM, system_prompt: str):
        self.card = card
        self.llm = llm_backend
        self.system_prompt = system_prompt
        self._tasks: Dict[str, A2ATask] = {}

    def get_agent_card(self) -> AgentCard:
        """A2A Discovery endpoint: GET /.well-known/agent.json"""
        return self.card

    def receive_task(self, user_message: str) -> A2ATask:
        """
        A2A Task Delegation endpoint: POST /tasks/send
        Spawns a task session, executes domain reasoning, and packages artifacts.
        """
        task = A2ATask()
        task.add_message(MessageRole.USER, user_message)
        task.state = TaskState.WORKING
        self._tasks[task.id] = task

        # Specialist prompt synthesis
        specialist_prompt = (
            f"=== SPECIALIST SYSTEM DIRECTIVE ===\n{self.system_prompt}\n\n"
            f"=== ASSIGNED TASK CONTEXT ===\n{user_message}\n\n"
            f"Provide your professional analysis and structured recommendations."
        )

        response = self.llm.generate(specialist_prompt)
        task.add_message(MessageRole.AGENT, response)
        task.complete(response)
        return task

    def get_task(self, task_id: str) -> Optional[A2ATask]:
        """A2A Polling endpoint: GET /tasks/{task_id}"""
        return self._tasks.get(task_id)


# ============================================================
# 🏛️ Instantiate Enterprise Architecture Specialist
# ============================================================

enterprise_arch_card = AgentCard(
    name="EnterpriseArchSpecialist",
    description="Specialist in distributed cloud architecture, microservices decomposition, and migration roadmaps.",
    url="agent://enterprise-arch-specialist/v1",
    version="1.0",
    capabilities=["streaming", "task-history"],
    skills=[
        "cloud-architecture-review",
        "strangler-fig-decomposition",
        "distributed-sql-design",
        "sla-capacity-planning"
    ]
)

enterprise_arch_agent = A2ASpecialistAgent(
    card=enterprise_arch_card,
    llm_backend=llm,
    system_prompt=(
        "You are a Principal Enterprise Cloud Architect. Your job is to analyze "
        "system requirements and design resilient, highly available cloud systems. "
        "Recommend proven architectural patterns (Strangler Fig, CQRS, Event-Driven) "
        "and specify infrastructure tiers with quantitative justifications."
    )
)

print("🏛️ Enterprise Architecture Specialist Created:")
enterprise_arch_agent.card.display()

In [ ]:
# ============================================================
# 🧪 In-Memory A2A Message Exchange Test
# ============================================================
print("=" * 60)
print("  🧪 A2A TEST: In-Memory Message Exchange")
print("=" * 60)

# Step 1: Agent Card Discovery
print("\n📡 Step 1: Inspecting Remote Agent Card...")
card = enterprise_arch_agent.get_agent_card()
print(f"   Agent Name: {card.name}")
print(f"   Skills: {', '.join(card.skills)}")

# Step 2: Task Delegation
print("\n📨 Step 2: Delegating task to EnterpriseArchSpecialist...")
delegation_payload = (
    "Client Scenario: Monolithic e-commerce platform handling 10k req/sec with frequent database locking. "
    "Goal: Migrate to cloud microservices with zero downtime and 99.999% data consistency."
)
task = enterprise_arch_agent.receive_task(delegation_payload)
print(f"   Task ID: {task.id}")
print(f"   Task State: {task.state.value}")

# Step 3: Inspect Artifacts
print("\n📋 Step 3: Inspecting specialist returned artifacts...")
result_text = task.artifacts[0]["content"]
print(f"   Specialist Report Snippet:\n   {result_text.strip()[:300]}...")
print("\n✅ A2A Delegation Exchange Completed Successfully!")

---
# 🎯 Section 4: THE ORCHESTRATION PIPELINE

Here we wire both protocols into a unified workforce:
1. **Facts Layer (MCP)**: Deterministic, bounded tool calls fetch ground truth facts.
2. **Specialist Layer (A2A)**: Open-ended, unbounded specialists analyze domain implications.
3. **Synthesis Layer (Orchestrator LLM)**: The central manager combines facts + specialist analysis into an executive strategy.

```
 User Request
      │
      ▼
 ┌─────────────────────────────────────────────────────────┐
 │                 🎯 ORCHESTRATOR                          │
 │                                                          │
 │  Phase 1: GATHER FACTS (MCP)                            │
 │  ┌─────────────────────────────────────────┐            │
 │  │  🔧 query_cloud_benchmarks("spanner")   │──► data   │
 │  │  🔧 query_cloud_benchmarks("cloud-run") │──► data   │
 │  └─────────────────────────────────────────┘            │
 │                         │                                │
 │                         ▼                                │
 │  Phase 2: SPECIALIST REASONING (A2A)                    │
 │  ┌─────────────────────────────────────────┐            │
 │  │  🤖 EnterpriseArchSpecialist            │            │
 │  │     Input:  Facts + Requirements        │            │
 │  │     Output: Architecture Blueprint      │──► report  │
 │  └─────────────────────────────────────────┘            │
 │                         │                                │
 │                         ▼                                │
 │  Phase 3: FINAL SYNTHESIS (LLM)                         │
 │  ┌─────────────────────────────────────────┐            │
 │  │  🧠 Central Orchestrator Synthesizer    │──► final   │
 │  └─────────────────────────────────────────┘            │
 └──────────────────────────┬───────────────────────────────┘
                            │
                            ▼
               📋 Final Executive Strategy
```

In [ ]:
# ============================================================
# 🎯 SECTION 4: Orchestrator Pipeline Implementation
# ============================================================

class OrchestratorAgent:
    """
    Central Orchestrator coordinating MCP tools and A2A specialists.
    """
    def __init__(self, llm_backend: BaseLLM, mcp_server: MCPToolServer):
        self.llm = llm_backend
        self.mcp = mcp_server
        self.specialists: Dict[str, A2ASpecialistAgent] = {}
        self.execution_trace: List[Dict[str, Any]] = []

    def register_specialist(self, specialist: A2ASpecialistAgent):
        card = specialist.get_agent_card()
        self.specialists[card.name] = specialist
        self._log("REGISTER", f"Registered A2A Specialist: {card.name}")

    def _log(self, stage: str, message: str, metadata: str = ""):
        self.execution_trace.append({
            "timestamp": time.time(),
            "stage": stage,
            "message": message,
            "metadata": metadata
        })

    def run(self, user_request: str) -> str:
        self.execution_trace = []
        t0 = time.time()

        print("╔" + "═" * 68 + "╗")
        print("║" + "  🚀 WORKFORCE PIPELINE EXECUTION TRACE".center(68) + "║")
        print("╠" + "═" * 68 + "╣")

        self._log("START", "Pipeline initialized with user request")

        # ─── Phase 1: Bounded Fact Gathering (MCP) ──────────────────────────
        print("║" + "  📥 Phase 1: GATHER FACTS (MCP Layer)".ljust(68) + "║")
        print("║" + "  " + "─" * 64 + "  ║")

        services_to_evaluate = ["cloud-spanner", "cloud-run"]
        benchmark_data = {}

        for service in services_to_evaluate:
            res = self.mcp.call_tool("query_cloud_benchmarks", {"service_name": service})
            benchmark_data[service] = res.get("data", res)
            if res.get("status") == "ok":
                d = res["data"]
                detail = f"p50: {d['latency_p50_ms']}ms | QPS: {d['throughput_qps']:,} | SLA: {d['availability_sla']}"
                print(f"║    🔧 MCP :: {service:<16} ➔ {detail:<38} ║")
                self._log("MCP_CALL", f"query_cloud_benchmarks({service})", detail)

        benchmark_summary = json.dumps(benchmark_data, indent=2)

        # ─── Phase 2: Unbounded Specialist Delegation (A2A) ─────────────────
        print("║" + "".center(68) + "║")
        print("║" + "  🤖 Phase 2: SPECIALIST DELEGATION (A2A Layer)".ljust(68) + "║")
        print("║" + "  " + "─" * 64 + "  ║")

        arch_specialist = self.specialists.get("EnterpriseArchSpecialist")
        arch_analysis = "No architecture specialist registered."

        if arch_specialist:
            arch_task_payload = (
                f"REQUIREMENTS:\n{user_request}\n\n"
                f"VERIFIED BENCHMARKS (from MCP):\n{benchmark_summary}"
            )
            task = arch_specialist.receive_task(arch_task_payload)
            arch_analysis = task.artifacts[0]["content"]
            print(f"║    📨 A2A :: EnterpriseArchSpecialist ➔ Task {task.id} [{task.state.value}]".ljust(69) + "║")
            self._log("A2A_DELEGATION", f"EnterpriseArchSpecialist Task={task.id}", arch_analysis[:120])

        # ─── Phase 3: Central Orchestrator Synthesis ────────────────────────
        print("║" + "".center(68) + "║")
        print("║" + "  🧠 Phase 3: FINAL SYNTHESIS (Central LLM)".ljust(68) + "║")
        print("║" + "  " + "─" * 64 + "  ║")

        synthesis_prompt = (
            f"You are the Lead Workforce Orchestrator.\n\n"
            f"USER OBJECTIVE:\n{user_request}\n\n"
            f"BENCHMARK FACTS (MCP Tool Layer):\n{benchmark_summary}\n\n"
            f"ARCHITECTURAL RECOMMENDATIONS (A2A Specialist Layer):\n{arch_analysis}\n\n"
            f"Synthesize a cohesive, prioritized executive migration strategy."
        )

        final_strategy = self.llm.generate(synthesis_prompt)
        print("║    ✅ Synthesis complete. Final strategy generated.".ljust(69) + "║")
        self._log("SYNTHESIS", "Executive strategy generated", final_strategy[:120])

        elapsed = time.time() - t0
        print("║" + "".center(68) + "║")
        print(f"║  ⏱️  Total Workforce Execution Time: {elapsed:.2f}s".ljust(69) + "║")
        print("╚" + "═" * 68 + "╝")

        return final_strategy

    def print_trace(self):
        print("\n📊 Execution Trace Breakdown:")
        print("─" * 60)
        t0 = self.execution_trace[0]["timestamp"]
        icons = {
            "START": "🏁", "REGISTER": "📝", "MCP_CALL": "🔧",
            "A2A_DELEGATION": "🤖", "SYNTHESIS": "🧠"
        }
        for item in self.execution_trace:
            delta = item["timestamp"] - t0
            icon = icons.get(item["stage"], "•")
            print(f"  [+{delta:04.2f}s] {icon} {item['stage']:<16} : {item['message']}")


print("✅ OrchestratorAgent class defined.")

In [ ]:
# ============================================================
# 🚀 Run the Complete Workforce Pipeline
# ============================================================

# 1. Initialize Orchestrator
orchestrator = OrchestratorAgent(llm_backend=llm, mcp_server=mcp_server)

# 2. Register Specialist
orchestrator.register_specialist(enterprise_arch_agent)

# 3. Define User Mission
mission = (
    "We need to migrate a high-volume payments & inventory system handling 10k RPS. "
    "The legacy monolith has severe database contention. We must achieve 99.99% uptime "
    "and zero data loss during cloud migration."
)

print(f"👤 Mission Prompt:\n   {mission}\n")

# 4. Execute
final_plan = orchestrator.run(mission)

# 5. Output Trace
orchestrator.print_trace()

# 6. Display Executive Strategy
print("\n" + "╔" + "═" * 68 + "╗")
print("║" + "  📋 FINAL SYNTHESIZED EXECUTIVE STRATEGY".center(68) + "║")
print("╠" + "═" * 68 + "╣")
for line in final_plan.split("\n"):
    for wrapped_line in textwrap.wrap(line, width=64) or [""]:
        print(f"║  {wrapped_line:<64}  ║")
print("╚" + "═" * 68 + "╝")

---
# 🏆 Section 5: HANDS-ON CHALLENGE (10 Minutes)

## Your Mission: Integrate a `SecurityAuditSpecialist`

In this challenge, you will expand the virtual workforce by implementing and wiring a **Security Specialist** into the pipeline.

### Target Architecture:

```
 Phase 1:  🔧 MCP Tools ──────────────► Benchmark Metrics
 Phase 2a: 🤖 EnterpriseArchSpecialist ► Architecture Blueprint
 Phase 2b: 🛡️  SecurityAuditSpecialist  ► Security & Threat Findings   ← [YOUR TASK]
 Phase 3:  🧠 LLM Synthesis ──────────► Final Hardened Cloud Strategy
```

### Instructions:
1. Complete **TODO 1**: Define the `SecurityAuditSpecialist` `AgentCard`.
2. Complete **TODO 2**: Create the `SecurityAuditSpecialist` agent instance with a security engineer system prompt.
3. Complete **TODO 3**: In `EnhancedOrchestrator`, wire Phase 2b to delegate the architecture plan to the security specialist.
4. Run the **Verification Cell** below to validate all 5 automated checks!

In [ ]:
# ============================================================
# 🏆 HANDS-ON CHALLENGE: Wire the Security Specialist
# ============================================================

# ── TODO 1: Complete the Security Agent Card ────────────────
# Define name, description, capabilities, and at least 3 security skills.

security_card = AgentCard(
    name="SecurityAuditSpecialist",
    description="Specialist in cloud security posture, threat modeling, zero-trust, and regulatory compliance.",
    url="agent://security-audit-specialist/v1",
    version="1.0",
    capabilities=["streaming", "threat-detection"],
    skills=[
        # TODO 1: Add 3+ security skills (e.g. 'threat-modeling', 'zero-trust-design', 'vulnerability-audit')
        "threat-modeling",
        "zero-trust-architecture",
        "vulnerability-audit",
        "compliance-verification"
    ]
)


# ── TODO 2: Instantiate the Security Specialist Agent ───────
# Supply the card, llm backend, and a detailed security persona system prompt.

security_agent = A2ASpecialistAgent(
    card=security_card,
    llm_backend=llm,
    system_prompt=(
        # TODO 2: Write a detailed system prompt for a Senior Security Architect
        "You are a Lead Cloud Security Architect. Your job is to rigorously audit "
        "proposed cloud architectures for vulnerabilities (OWASP Top 10, lateral movement, "
        "missing mTLS, unencrypted egress, secret exposure). Provide concrete risk ratings and remediations."
    )
)


# ── TODO 3: Implement Phase 2b in the Enhanced Orchestrator ──

class EnhancedOrchestrator(OrchestratorAgent):
    """
    Enhanced Orchestrator wiring both Architecture and Security specialists.
    """
    def run_with_security(self, user_request: str) -> tuple[str, str, str]:
        self.execution_trace = []
        t0 = time.time()

        print("╔" + "═" * 68 + "╗")
        print("║" + "  🛡️  ENHANCED MULTI-SPECIALIST PIPELINE RUN".center(68) + "║")
        print("╠" + "═" * 68 + "╣")

        # Phase 1: MCP Facts
        print("║" + "  📥 Phase 1: GATHER FACTS (MCP Layer)".ljust(68) + "║")
        services = ["cloud-spanner", "cloud-run"]
        benchmarks = {}
        for s in services:
            res = self.mcp.call_tool("query_cloud_benchmarks", {"service_name": s})
            benchmarks[s] = res.get("data", res)
            print(f"║    🔧 MCP :: {s:<16} ➔ OK".ljust(69) + "║")
        benchmarks_json = json.dumps(benchmarks, indent=2)

        # Phase 2a: Architecture Specialist
        print("║" + "".center(68) + "║")
        print("║" + "  🤖 Phase 2a: ARCHITECTURE REASONING (A2A)".ljust(68) + "║")
        arch = self.specialists.get("EnterpriseArchSpecialist")
        arch_findings = "(Architecture specialist not registered)"
        if arch:
            t_arch = arch.receive_task(f"Request: {user_request}\nBenchmarks:\n{benchmarks_json}")
            arch_findings = t_arch.artifacts[0]["content"]
            print(f"║    📨 A2A :: EnterpriseArchSpecialist ➔ Task {t_arch.id} [completed]".ljust(69) + "║")

        # Phase 2b: Security Specialist [YOUR IMPLEMENTATION HERE]
        print("║" + "".center(68) + "║")
        print("║" + "  🛡️  Phase 2b: SECURITY & THREAT AUDIT (A2A)".ljust(68) + "║")

        # ====================================================================
        # TODO 3: Retrieve 'SecurityAuditSpecialist' from self.specialists,
        # delegate a task containing the architecture blueprint, and extract
        # the security findings artifact.
        # ====================================================================
        sec = self.specialists.get("SecurityAuditSpecialist")
        if sec:
            t_sec = sec.receive_task(
                f"AUDIT PROPOSED ARCHITECTURE FOR RISKS:\n\n"
                f"Proposed Architecture:\n{arch_findings}"
            )
            security_findings = t_sec.artifacts[0]["content"]
            print(f"║    🛡️  A2A :: SecurityAuditSpecialist ➔ Task {t_sec.id} [completed]".ljust(69) + "║")
        else:
            security_findings = "(Security specialist not registered)"

        # Phase 3: Unified Synthesis
        print("║" + "".center(68) + "║")
        print("║" + "  🧠 Phase 3: HARDENED SYNTHESIS (Central LLM)".ljust(68) + "║")
        synthesis_prompt = (
            f"Synthesize an Executive Cloud Migration Strategy incorporating both "
            f"Architecture and Security assessments:\n\n"
            f"User Goal: {user_request}\n\n"
            f"Architecture Blueprint:\n{arch_findings}\n\n"
            f"Security Audit Findings:\n{security_findings}\n\n"
            f"Provide an actionable, secure migration plan."
        )
        final_strategy = self.llm.generate(synthesis_prompt)
        print("║    ✅ Synthesis Complete.".ljust(69) + "║")

        elapsed = time.time() - t0
        print("║" + "".center(68) + "║")
        print(f"║  ⏱️  Total Workforce Run Time: {elapsed:.2f}s".ljust(69) + "║")
        print("╚" + "═" * 68 + "╝")

        return final_strategy, arch_findings, security_findings


print("✅ Challenge implementation ready. Run the verification cell below!")

In [ ]:
# ============================================================
# ✅ VERIFICATION SUITE: Run to test your workforce!
# ============================================================

def verify_challenge():
    print("=" * 68)
    print("  🔍 RUNNING AUTOMATED CHALLENGE VERIFICATION")
    print("=" * 68)

    checks = []

    # Check 1: Agent Card
    print("\n[Check 1/5] Validating Security Agent Card...")
    if security_card.name == "SecurityAuditSpecialist" and "TODO" not in security_card.description:
        if len(security_card.skills) >= 3:
            print(f"  ✅ Agent Card configured with skills: {security_card.skills}")
            checks.append(True)
        else:
            print("  ❌ Agent Card must define at least 3 skills.")
            checks.append(False)
    else:
        print("  ❌ Agent Card name or description invalid.")
        checks.append(False)

    # Check 2: System Prompt
    print("\n[Check 2/5] Validating Security Specialist Persona...")
    if "TODO" not in security_agent.system_prompt and len(security_agent.system_prompt) >= 40:
        print(f"  ✅ System Prompt valid ({len(security_agent.system_prompt)} characters)")
        checks.append(True)
    else:
        print("  ❌ System prompt is too short or contains TODO.")
        checks.append(False)

    # Check 3: Specialist Registration
    print("\n[Check 3/5] Testing Orchestrator Registration...")
    enhanced_orch = EnhancedOrchestrator(llm_backend=llm, mcp_server=mcp_server)
    enhanced_orch.register_specialist(enterprise_arch_agent)
    enhanced_orch.register_specialist(security_agent)

    if "SecurityAuditSpecialist" in enhanced_orch.specialists:
        print("  ✅ SecurityAuditSpecialist successfully registered in workforce registry.")
        checks.append(True)
    else:
        print("  ❌ SecurityAuditSpecialist missing from registry.")
        checks.append(False)

    # Check 4: Pipeline Execution & Security findings
    print("\n[Check 4/5] Executing Enhanced Multi-Agent Pipeline...")
    try:
        strat, arch_res, sec_res = enhanced_orch.run_with_security(
            "Migrate core payments platform to cloud with zero downtime."
        )
        if sec_res and "TODO" not in sec_res and len(sec_res) > 30:
            print("  ✅ Phase 2b successfully executed and returned findings.")
            checks.append(True)
        else:
            print("  ❌ Phase 2b returned empty or invalid security findings.")
            checks.append(False)
    except Exception as e:
        print(f"  ❌ Pipeline execution failed with error: {e}")
        checks.append(False)

    # Check 5: Hardened Synthesis
    print("\n[Check 5/5] Validating Final Synthesized Strategy...")
    if strat and len(strat) > 50:
        print("  ✅ Executive hardened strategy successfully synthesized.")
        checks.append(True)
    else:
        print("  ❌ Final strategy output was empty.")
        checks.append(False)

    # Summary
    print("\n" + "=" * 68)
    passed = sum(checks)
    total = len(checks)
    if passed == total:
        print(f"  🎉 PERFECT SCORE! {passed}/{total} CHECKS PASSED.")
        print("  You have built a fully functional MCP + A2A Virtual Workforce!")
    else:
        print(f"  ⚠️  {passed}/{total} checks passed. Please review the errors above.")
    print("=" * 68)


verify_challenge()

---
# 🎓 Summary & Architectural Takeaways

Congratulations on building a mini virtual workforce! Here is the mental framework to take into production:

| Component | Standard | Boundary Nature | Role in System |
|---|---|---|---|
| `query_cloud_benchmarks` | **MCP** | **Bounded** | Deterministic, stateless fact retrieval |
| `EnterpriseArchSpecialist` | **A2A** | **Unbounded** | Open-ended architectural reasoning |
| `SecurityAuditSpecialist` | **A2A** | **Unbounded** | Autonomous threat modeling & audit |
| `OrchestratorAgent` | **Multi-Protocol** | **Coordinator** | Workflow execution & synthesis |

### Key Architectural Rules of Thumb:
1. **Never make an LLM do what a deterministic tool can do** — use MCP for DBs, APIs, and calculators.
2. **Never overload a single prompt with multiple specialist domains** — use A2A delegation to keep context bounded.
3. **Always use machine-readable Agent Cards** for discoverability and loose coupling.

---
### 🔗 Next Steps & Further Reading
- [Model Context Protocol (MCP) Official Spec](https://modelcontextprotocol.io/)
- [Agent-to-Agent (A2A) Protocol Specification](https://google.github.io/A2A/)
- [Google GenAI SDK Documentation](https://cloud.google.com/vertex-ai/generative-ai/docs/reference/python/latest)